In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# load
df = pd.read_csv("../models/clustered_data.csv")
embeddings = np.load("../models/embeddings.npy")

# remove outliers
mask = df.cluster != -1

X = embeddings[mask]
y = df.cluster[mask]

print("Samples:", len(X))
print("Clusters:", len(set(y)))

# metrics
sil = silhouette_score(X, y)
db = davies_bouldin_score(X, y)
ch = calinski_harabasz_score(X, y)

print("\nRESULTS")
print("Silhouette Score:", round(sil,4))
print("Davies Bouldin:", round(db,4))
print("Calinski Harabasz:", round(ch,2))

Samples: 81144
Clusters: 20

RESULTS
Silhouette Score: -0.0313
Davies Bouldin: 3.017
Calinski Harabasz: 334.16


In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import time

model = SentenceTransformer("../models/sentence_model")

centroids = []
clusters = sorted([c for c in df.cluster.unique() if c!=-1])

for c in clusters:
    idx = df[df.cluster==c].index
    centroids.append(embeddings[idx].mean(axis=0))

centroids = np.array(centroids)

test = "I feel anxious all the time"

start = time.time()

vec = model.encode([test])[0]
sim = cosine_similarity([vec], centroids)
pred = np.argmax(sim)

end = time.time()

print("Prediction time:", round((end-start)*1000,2), "ms")

/Users/krish/Downloads/MindScope/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5412.88it/s]


Prediction time: 3755.54 ms


In [3]:
start = time.time()

for _ in range(50):
    vec = model.encode([test])[0]
    sim = cosine_similarity([vec], centroids)
    pred = np.argmax(sim)

end = time.time()

avg = ((end-start)/50)*1000
print("Average:", round(avg,2), "ms")

Average: 30.98 ms
